In [22]:
import ot
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
import math
import scipy
from scipy.spatial import cKDTree
from scipy import integrate
from scipy import interpolate
from scipy.spatial import Delaunay
from scipy.spatial import Voronoi
from itertools import product
import torch
torch.manual_seed(100000)
import torch.nn as nn
import torch.optim as optim


In [23]:
#settings
N = 5  # Number of variables
F = 8  # Forcing
start = F * np.ones(N)  # Initial state (equilibrium)
start[0] += 0.01  # Add small perturbation to the first variable
dt = 0.01
subset_size = 100#cell number
traj_length = int(1e6)#length of trajectory
noise_level = 0.2
sample_size = int(5000)
slope = 0.1
simulation_t = int(5e3)
epsilon = 1e-4
steps = 5
N_iters = 20000

costs1W, costs2W, costs1L, costs2L = [],[],[],[]
def L96(x, t):
    """Lorenz 96 model with constant forcing"""
    return (np.roll(x, -1) - np.roll(x, 2)) * np.roll(x, 1) - x + F 
def L96_vec(X):

    return (torch.roll(X, shifts=-1, dims=1) - torch.roll(X, shifts=2, dims=1)) * torch.roll(X, shifts=1, dims=1) - X + F

t = np.arange(0.0, traj_length*dt, dt)
relu = nn.ReLU()
def decay(x):
    return relu(1-slope*x)
def w(xs):
    dists =  torch.cdist(xs, Voronoi_centers, p =2)
    pre_w = decay(dists)
    return  pre_w / torch.sum(pre_w, dim=1,keepdim = True)


def Ulam(points,Tpoints):#column stochastic
    mat = torch.zeros((subset_size, subset_size))
    #before normalization
    with torch.no_grad():
          randpts_idxs = torch.tensor(tree.query(points.detach().numpy())[1], dtype=torch.int)
    weights = w(Tpoints)
    mat.index_add_(0, randpts_idxs, weights)
    mat = (mat.T)/mat.sum(dim = 1)
    return mat

class W2Loss(torch.autograd.Function):
    @staticmethod
    def forward(ctx, U_net):
        U_net_np = U_net.detach().numpy()
        cost_cols,grad = np.zeros(subset_size),np.zeros((subset_size,subset_size))
        costM = ot.dist(np.arange(subset_size).reshape(-1, 1), np.arange(subset_size).reshape(-1, 1))
        for col in range(subset_size):
            _, log = ot.emd(U_true_np[:,col], U_net_np[:,col], costM, log=True)
            cost_cols[col],grad[col] = log['cost'],log["v"]
        loss,grad = np.sum(cost_cols),grad.T
        grad_tensor = torch.tensor(grad, dtype=U_net.dtype)
        ctx.save_for_backward(grad_tensor)
        return torch.tensor(loss, dtype=U_net.dtype)
    @staticmethod
    def backward(ctx, grad_output):
        grad_tensor, = ctx.saved_tensors  # Unpack gradient
        return grad_tensor.reshape(subset_size,subset_size)

In [ ]:
trajectory_clean = scipy.integrate.odeint(L96, start, t)
trajectory = trajectory_clean + np.random.normal(0,noise_level,((traj_length,5)))#loooooooong trajectory
trajectory = trajectory[int(1e3):]

In [ ]:
#uniformly random data
    
rand_idxs = np.random.choice(len(trajectory), size=sample_size, replace=False)
observed = trajectory[rand_idxs]
randpts = torch.tensor(trajectory[rand_idxs],dtype = torch.float)
Trandpts = torch.tensor(trajectory[rand_idxs+steps],dtype = torch.float)

#normalization
M_scale = torch.max(torch.abs(randpts))
randpts /= M_scale
Trandpts /= M_scale

#Voronoi cell center
Voronoi_centers = MiniBatchKMeans(n_clusters=subset_size).fit(randpts).cluster_centers_

tree = cKDTree(Voronoi_centers)
Voronoi_centers = torch.tensor(Voronoi_centers,dtype = torch.float)

U_true = Ulam(randpts,Trandpts)
U_true_np = U_true.detach().cpu().numpy()

In [ ]:
for iteration in range(10):
    net1 = nn.Sequential(
            nn.Linear(5, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 5))

    optimizer1 = optim.Adam(net1.parameters(),lr = 1e-3)

    net1.train()
    loss1 = []
    # Store the first loss
    net1_randpts = randpts.clone()  # safe copy, avoid modifying original tensor
    for _ in range(steps):
        V_field = net1(net1_randpts)
        net1_randpts = net1_randpts + dt * V_field  # out-of-place update
    U_net = Ulam(randpts, net1_randpts)
    initial_L1 = W2Loss.apply(U_net)
    for i in range(N_iters):
        optimizer1.zero_grad()
        net1_randpts = randpts
        for _ in range(steps):
            V_field = net1(net1_randpts)
            net1_randpts = net1_randpts+dt* V_field
        U_net = Ulam(randpts,net1_randpts)
        L1 = W2Loss.apply(U_net)
        L1.backward()
        optimizer1.step()
        loss1.append(L1.item())
        if i % 500 == 0:
            if L1.item() < 0.02 * initial_L1:
                break
    x1 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, N)
    vals1 = [x1.detach().numpy().flatten()]
    for _ in range(simulation_t-1):

        V_field = net1(x1)
        x1 = x1+dt* V_field
        vals1.append(x1.detach().numpy().flatten())
    vals1 = np.array(vals1)
    net2 = nn.Sequential(
            nn.Linear(5, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 5))


    optimizer2 = optim.Adam(net2.parameters(),lr = 1e-3)
    net2.train()
    loss2 = []
    net2_randpts = randpts.clone()  # safe copy, avoid modifying original tensor
    for _ in range(steps):
        V_field2 = net2(net2_randpts)
        net2_randpts = net2_randpts + dt * V_field2  # out-of-place update
    initial_L2 = torch.mean((net2_randpts - Trandpts) ** 2)
    for i in range(N_iters):
        optimizer2.zero_grad()
        net2_randpts = randpts
        for _ in range(steps):
            V_field2 = net2(net2_randpts)
            net2_randpts = net2_randpts+dt* V_field2
        L2 = torch.mean((net2_randpts - Trandpts) ** 2)
        L2.backward()
        optimizer2.step()
        loss2.append(L2.item())
        if i % 500 == 0:
            if L2.item() < 0.02 * initial_L2:
                break
    x2 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 5)
    vals2 = [x2.detach().numpy().flatten()]
    for _ in range(simulation_t-1):

        V_field2 = net2(x2)
        x2 = x2 + V_field2*dt
        vals2.append(x2.detach().numpy().flatten())
    vals2 = np.array(vals2)
    GT = scipy.integrate.odeint(L96, M_scale*randpts[0], np.arange(0.0, simulation_t*dt, dt))
    
    
    subGT, subvals1, subvals2 = GT, np.array(M_scale * vals1), np.array(M_scale * vals2)
    a, b = np.ones((simulation_t,)) / simulation_t, np.ones((simulation_t,)) / simulation_t
    M1, M2 = ot.dist(subGT, subvals1),ot.dist(subGT, subvals2)
    _,G01 = ot.emd(a, b, M1,log = 'true')
    _,G02 = ot.emd(a, b, M2,log = 'true')
    cost1W,cost2W = (G01['cost'])**(1/2),(G02['cost'])**(1/2)
    GTpts = torch.tensor((GT/M_scale)[:-steps],dtype = torch.float)
    GTTpts = GT[steps:]

    net1pts,net2pts = GTpts, GTpts
    for i in range(steps):
        Vf1,Vf2 = net1(net1pts),net2(net2pts)
        net1pts,net2pts = net1pts+dt*Vf1,net2pts+dt*Vf2
    cost1L = (np.mean((np.array(M_scale)*net1pts.detach().numpy()-GTTpts)**2))**(1/2)
    cost2L = (np.mean((np.array(M_scale)*net2pts.detach().numpy()-GTTpts)**2))**(1/2)
    print("Experiment ", iteration,": W2(ours) ",cost1W, ", W2(pointwise) ",cost2W)
    print("Experiment ", iteration,": L2(ours) ",cost1L, ", L2(pointwise) ",cost2L)
    costs1W.append(cost1W)
    costs2W.append(cost2W)
    costs1L.append(cost1L)
    costs2L.append(cost2L)

In [ ]:
costs1W,costs2W,costs1L,costs2L

In [ ]:
np.mean(costs1W),np.std(costs1W)

In [ ]:
np.mean(costs2W),np.std(costs2W)

In [ ]:
np.mean(costs1L),np.std(costs1L)

In [ ]:
np.mean(costs2L),np.std(costs2L)